# 🚲 Notebook Colab — Personas enrichis × Mobilité EMP 2019

## À quoi sert ce notebook ?

Ce notebook est le **compagnon du simulateur Vélis** disponible sur Streamlit.
Il permet de passer des estimations statistiques intégrées par défaut aux **données réelles**
calculées sur les 6 millions de personas du dataset NVIDIA Nemotron-Personas-France.

Il produit deux fichiers à uploader dans l'application :

| Fichier | Taille | Utilisation |
|---|---|---|
| `personas_mobilite_200k.csv` | ~166 Mo · 200 000 lignes | **Onglet 3 (Simulateur personas)** — remplace les 15 personas de démo |
| `distributions_nemotron.json` | < 1 Mo | **Sidebar (Onglet 1 Optimiseur)** — recalibre les courbes de mobilité |

---

## Comment ça fonctionne ?

### Étapes 1–8 : Génération du CSV (200 000 personas enrichis)

Le notebook se connecte au dataset Nemotron sur HuggingFace et scanne les profils **en streaming**
(sans télécharger les 6 millions d'un coup). Pour chaque persona, il applique deux filtres :

**Filtre 1 — Faibles revenus** : l'occupation est de type ouvrier, employé, retraité, artisan,
agriculteur, aide-soignant, livreur… Les cadres supérieurs, médecins, directeurs sont exclus.

**Filtre 2 — Territoire rural** : le département du persona est classé "rural" ou "rural isolé"
selon un mapping basé sur la densité communale INSEE.
*Note : dans l'échantillon actuel, le filtre Q1+Q2 de Nemotron ne produit que des ruraux —
les périurbains sont absents de cet échantillon low-income.*

Pour chaque persona retenu, le notebook attribue des **statistiques de mobilité individuelles**
en puisant dans une table de référence de 35 segments (territoire × CSP) extraits des
micro-données EMP 2019 (45 169 déplacements individuels réels). Il ajoute un bruit log-normal
calibré pour que les distributions individuelles correspondent aux distributions observées.

**Les 25 colonnes du CSV de sortie :**

```
Identité :     uuid · prenom · age · sex · commune · departement · territoire · occupation · csp
               household_type · education_level
Profil :       sport · travel · skills · hobbies
Budget :       budget_achat_eur
Mobilité :     emp_segment · km_jour · nb_deplacements · dist_max_trajet
               besoin_autonomie_km · vitesse_min_kmh · places_besoin
CO₂ :          co2_actuel_g_jour · co2_economie_g_jour
```

**Résultats observés sur la version actuelle :**
- 200 000 personas collectés / 213 449 scannés
- 100 % ruraux (rural · famille 72 % · rural · navetteur 25 % · rural · longue dist. 3 %)
- km/jour médian : 31 km · Autonomie P50 : 24 km · P75 : 40 km · P90 : 66 km
- Budget médian : 8 500 € · CO₂ actuel médian : 5 425 gCO₂/personne/jour

### Étapes 9a–9d : Calcul des distributions pour l'Optimiseur

Sur le CSV généré, le notebook calcule pour chacun des 6 profils archétypaux
les **distributions cumulatives** qui alimentent l'onglet 1 "Optimiseur des aides" :

**CUMUL_AUTO** = P(besoin_autonomie_km ≤ X) : quelle fraction du profil est couverte
par un véhicule d'autonomie X km ?
*Exemple : 75 % des rural_navetteur ont besoin de ≤ 30 km d'autonomie*

**AFFORD_PROFILES** = P(budget_achat_eur ≥ prix) : quelle fraction du profil peut
financer un véhicule au prix net donné ?
*Exemple : 48 % des rural_navetteur peuvent financer 8 000 € net*

Ces distributions remplacent les estimations CGDD/INSEE intégrées par défaut.

**Si un profil a 0 personas dans l'échantillon** (typiquement les 3 profils périurbains),
l'app conserve automatiquement les valeurs EMP 2019 pour ce profil et affiche un avertissement.
Ce n'est pas un bug — c'est une limite de l'échantillon Nemotron actuel.

---

## Prérequis et durée

- **Compte Google** (Colab gratuit)
- **Aucune installation, aucun téléchargement préalable**
- **Durée** : ~20 min (2M personas scannés, GPU non requis)

Pour aller plus vite : réduire `SCAN_MAX` à 500 000 (cellule 2) — ~5 min, moins de personas.
Pour plus de diversité : augmenter `SCAN_MAX` à 4 000 000 — ~40 min.

---

## Utilisation des fichiers produits

### `personas_mobilite_200k.csv` → Onglet 3 (Simulateur personas)

```
App Streamlit → Onglet 🔍 Simulateur personas
→ Expander "📥 Charger des personas depuis Colab"
→ Browse files → personas_mobilite_200k.csv
```
Les 15 personas de démonstration sont remplacés par vos 200 000 personas Nemotron enrichis.
Vous pouvez ensuite explorer n'importe quel profil et voir quel véhicule Vélis lui correspond.

### `distributions_nemotron.json` → Sidebar (Onglet 1 Optimiseur)

```
App Streamlit → Sidebar gauche → "📥 Données Nemotron avancées"
→ Browse files → distributions_nemotron.json
```
L'onglet 1 se recalibre immédiatement. Un message de confirmation indique quels profils
ont été chargés depuis Nemotron et lesquels conservent les valeurs EMP 2019 par défaut.


In [ ]:
# @title ⚙️ 1. Installation
!pip install datasets huggingface_hub pandas tqdm -q
print('✅ Installation terminée')

In [ ]:
# @title ⚙️ 2. Paramètres
N_TARGET    = 200_000  # @param {type:'integer'}
SCAN_MAX    = 2_000_000  # @param {type:'integer'}
OUTPUT_FILE = 'personas_mobilite_200k.csv'  # @param {type:'string'}
SEED        = 42  # @param {type:'integer'}
import numpy as np
np.random.seed(SEED)
print(f'Objectif : {N_TARGET:,} personas · scan max {SCAN_MAX:,} lignes')

In [ ]:
# @title 📊 3. Référentiel mobilité EMP 2019 (données réelles microdata)
# Issu des micro-données EMP 2019 — déplacements semaine, low-income (Q1+Q2)
# Segments : territoire × CSP
# Indicateurs par individu : km/jour total, nb déplacements, dist_max_trajet (≈ besoin autonomie)
import json

EMP_REF = {
  "periurbain__agriculteurs": {
    "n": 1,
    "km_jour_moy": 2.0,
    "km_jour_med": 2.0,
    "km_jour_p75": 2.0,
    "km_jour_p90": 2.0,
    "nb_dep_moy": 2.0,
    "nb_dep_med": 2.0,
    "autonomie_moy": 1.0,
    "autonomie_med": 1.0,
    "autonomie_p75": 1.0,
    "autonomie_p90": 1.0,
    "co2_moy": 372.0
  },
  "periurbain__artisans": {
    "n": 25,
    "km_jour_moy": 68.3,
    "km_jour_med": 25.8,
    "km_jour_p75": 71.0,
    "km_jour_p90": 156.1,
    "nb_dep_moy": 4.0,
    "nb_dep_med": 4.0,
    "autonomie_moy": 41.3,
    "autonomie_med": 12.0,
    "autonomie_p75": 30.0,
    "autonomie_p90": 67.1,
    "co2_moy": 5169.7
  },
  "periurbain__autre": {
    "n": 66,
    "km_jour_moy": 10.9,
    "km_jour_med": 3.7,
    "km_jour_p75": 9.8,
    "km_jour_p90": 30.5,
    "nb_dep_moy": 3.3,
    "nb_dep_med": 3.0,
    "autonomie_moy": 6.0,
    "autonomie_med": 1.5,
    "autonomie_p75": 3.2,
    "autonomie_p90": 11.5,
    "co2_moy": 762.5
  },
  "periurbain__cadres": {
    "n": 19,
    "km_jour_moy": 50.2,
    "km_jour_med": 20.5,
    "km_jour_p75": 41.8,
    "km_jour_p90": 128.4,
    "nb_dep_moy": 3.3,
    "nb_dep_med": 3.0,
    "autonomie_moy": 26.3,
    "autonomie_med": 10.7,
    "autonomie_p75": 20.1,
    "autonomie_p90": 60.0,
    "co2_moy": 5537.6
  },
  "periurbain__employes": {
    "n": 168,
    "km_jour_moy": 36.7,
    "km_jour_med": 18.9,
    "km_jour_p75": 40.0,
    "km_jour_p90": 83.0,
    "nb_dep_moy": 4.0,
    "nb_dep_med": 4.0,
    "autonomie_moy": 18.0,
    "autonomie_med": 6.3,
    "autonomie_p75": 15.0,
    "autonomie_p90": 30.6,
    "co2_moy": 3338.1
  },
  "periurbain__inactifs": {
    "n": 177,
    "km_jour_moy": 21.2,
    "km_jour_med": 8.0,
    "km_jour_p75": 18.0,
    "km_jour_p90": 61.5,
    "nb_dep_moy": 2.9,
    "nb_dep_med": 2.0,
    "autonomie_moy": 9.4,
    "autonomie_med": 3.5,
    "autonomie_p75": 8.0,
    "autonomie_p90": 25.0,
    "co2_moy": 2094.1
  },
  "periurbain__intermediaires": {
    "n": 96,
    "km_jour_moy": 44.8,
    "km_jour_med": 20.7,
    "km_jour_p75": 57.2,
    "km_jour_p90": 109.0,
    "nb_dep_moy": 3.5,
    "nb_dep_med": 3.0,
    "autonomie_moy": 25.8,
    "autonomie_med": 7.5,
    "autonomie_p75": 27.0,
    "autonomie_p90": 60.0,
    "co2_moy": 4457.7
  },
  "periurbain__ouvriers": {
    "n": 178,
    "km_jour_moy": 45.0,
    "km_jour_med": 17.0,
    "km_jour_p75": 43.5,
    "km_jour_p90": 98.6,
    "nb_dep_moy": 3.6,
    "nb_dep_med": 3.0,
    "autonomie_moy": 23.7,
    "autonomie_med": 7.0,
    "autonomie_p75": 18.4,
    "autonomie_p90": 39.3,
    "co2_moy": 3775.7
  },
  "periurbain__retraites": {
    "n": 569,
    "km_jour_moy": 24.9,
    "km_jour_med": 8.0,
    "km_jour_p75": 23.6,
    "km_jour_p90": 60.0,
    "nb_dep_moy": 3.0,
    "nb_dep_med": 2.0,
    "autonomie_moy": 13.7,
    "autonomie_med": 4.0,
    "autonomie_p75": 10.0,
    "autonomie_p90": 30.0,
    "co2_moy": 2019.9
  },
  "rural__agriculteurs": {
    "n": 25,
    "km_jour_moy": 30.8,
    "km_jour_med": 12.4,
    "km_jour_p75": 38.6,
    "km_jour_p90": 67.9,
    "nb_dep_moy": 4.8,
    "nb_dep_med": 4.0,
    "autonomie_moy": 13.3,
    "autonomie_med": 5.0,
    "autonomie_p75": 15.0,
    "autonomie_p90": 34.9,
    "co2_moy": 4732.9
  },
  "rural__artisans": {
    "n": 36,
    "km_jour_moy": 39.9,
    "km_jour_med": 37.0,
    "km_jour_p75": 54.0,
    "km_jour_p90": 76.3,
    "nb_dep_moy": 3.8,
    "nb_dep_med": 4.0,
    "autonomie_moy": 17.1,
    "autonomie_med": 12.0,
    "autonomie_p75": 26.0,
    "autonomie_p90": 36.0,
    "co2_moy": 5774.1
  },
  "rural__autre": {
    "n": 58,
    "km_jour_moy": 26.0,
    "km_jour_med": 15.7,
    "km_jour_p75": 34.8,
    "km_jour_p90": 55.0,
    "nb_dep_moy": 3.2,
    "nb_dep_med": 2.0,
    "autonomie_moy": 13.2,
    "autonomie_med": 8.3,
    "autonomie_p75": 15.0,
    "autonomie_p90": 32.3,
    "co2_moy": 1892.6
  },
  "rural__cadres": {
    "n": 18,
    "km_jour_moy": 68.1,
    "km_jour_med": 52.0,
    "km_jour_p75": 74.2,
    "km_jour_p90": 121.8,
    "nb_dep_moy": 3.5,
    "nb_dep_med": 2.5,
    "autonomie_moy": 37.5,
    "autonomie_med": 27.1,
    "autonomie_p75": 36.0,
    "autonomie_p90": 47.9,
    "co2_moy": 6798.6
  },
  "rural__employes": {
    "n": 145,
    "km_jour_moy": 44.5,
    "km_jour_med": 34.0,
    "km_jour_p75": 60.0,
    "km_jour_p90": 88.0,
    "nb_dep_moy": 3.7,
    "nb_dep_med": 3.0,
    "autonomie_moy": 19.8,
    "autonomie_med": 15.0,
    "autonomie_p75": 27.0,
    "autonomie_p90": 40.0,
    "co2_moy": 5473.9
  },
  "rural__inactifs": {
    "n": 100,
    "km_jour_moy": 34.2,
    "km_jour_med": 20.0,
    "km_jour_p75": 40.0,
    "km_jour_p90": 70.3,
    "nb_dep_moy": 2.9,
    "nb_dep_med": 2.0,
    "autonomie_moy": 17.7,
    "autonomie_med": 9.5,
    "autonomie_p75": 20.0,
    "autonomie_p90": 40.0,
    "co2_moy": 3062.2
  },
  "rural__intermediaires": {
    "n": 89,
    "km_jour_moy": 80.5,
    "km_jour_med": 50.0,
    "km_jour_p75": 93.0,
    "km_jour_p90": 172.8,
    "nb_dep_moy": 4.0,
    "nb_dep_med": 4.0,
    "autonomie_moy": 40.4,
    "autonomie_med": 20.0,
    "autonomie_p75": 36.0,
    "autonomie_p90": 94.1,
    "co2_moy": 7950.6
  },
  "rural__ouvriers": {
    "n": 171,
    "km_jour_moy": 47.7,
    "km_jour_med": 32.5,
    "km_jour_p75": 57.6,
    "km_jour_p90": 100.0,
    "nb_dep_moy": 3.6,
    "nb_dep_med": 3.0,
    "autonomie_moy": 22.1,
    "autonomie_med": 14.0,
    "autonomie_p75": 24.5,
    "autonomie_p90": 45.0,
    "co2_moy": 5315.9
  },
  "rural__retraites": {
    "n": 629,
    "km_jour_moy": 34.2,
    "km_jour_med": 17.5,
    "km_jour_p75": 35.0,
    "km_jour_p90": 73.5,
    "nb_dep_moy": 3.0,
    "nb_dep_med": 2.0,
    "autonomie_moy": 18.8,
    "autonomie_med": 7.5,
    "autonomie_p75": 15.0,
    "autonomie_p90": 33.1,
    "co2_moy": 2890.5
  },
  "rural_isole__agriculteurs": {
    "n": 20,
    "km_jour_moy": 32.1,
    "km_jour_med": 22.3,
    "km_jour_p75": 60.3,
    "km_jour_p90": 74.3,
    "nb_dep_moy": 3.6,
    "nb_dep_med": 3.0,
    "autonomie_moy": 15.0,
    "autonomie_med": 9.9,
    "autonomie_p75": 19.5,
    "autonomie_p90": 35.8,
    "co2_moy": 4445.1
  },
  "rural_isole__artisans": {
    "n": 2,
    "km_jour_moy": 135.9,
    "km_jour_med": 135.9,
    "km_jour_p75": 158.8,
    "km_jour_p90": 172.5,
    "nb_dep_moy": 4.0,
    "nb_dep_med": 4.0,
    "autonomie_moy": 59.5,
    "autonomie_med": 59.5,
    "autonomie_p75": 66.8,
    "autonomie_p90": 71.1,
    "co2_moy": 26308.5
  },
  "rural_isole__autre": {
    "n": 5,
    "km_jour_moy": 23.9,
    "km_jour_med": 25.3,
    "km_jour_p75": 26.0,
    "km_jour_p90": 27.7,
    "nb_dep_moy": 2.8,
    "nb_dep_med": 2.0,
    "autonomie_moy": 12.9,
    "autonomie_med": 13.0,
    "autonomie_p75": 15.0,
    "autonomie_p90": 18.9,
    "co2_moy": 1288.6
  },
  "rural_isole__cadres": {
    "n": 3,
    "km_jour_moy": 32.8,
    "km_jour_med": 10.0,
    "km_jour_p75": 45.5,
    "km_jour_p90": 66.8,
    "nb_dep_moy": 2.3,
    "nb_dep_med": 2.0,
    "autonomie_moy": 13.6,
    "autonomie_med": 5.0,
    "autonomie_p75": 18.5,
    "autonomie_p90": 26.6,
    "co2_moy": 4591.7
  },
  "rural_isole__employes": {
    "n": 18,
    "km_jour_moy": 52.9,
    "km_jour_med": 42.9,
    "km_jour_p75": 79.8,
    "km_jour_p90": 107.0,
    "nb_dep_moy": 3.3,
    "nb_dep_med": 3.0,
    "autonomie_moy": 23.6,
    "autonomie_med": 19.0,
    "autonomie_p75": 39.2,
    "autonomie_p90": 49.5,
    "co2_moy": 6928.8
  },
  "rural_isole__inactifs": {
    "n": 14,
    "km_jour_moy": 44.1,
    "km_jour_med": 37.5,
    "km_jour_p75": 50.7,
    "km_jour_p90": 88.9,
    "nb_dep_moy": 3.5,
    "nb_dep_med": 3.5,
    "autonomie_moy": 21.9,
    "autonomie_med": 16.0,
    "autonomie_p75": 34.6,
    "autonomie_p90": 45.0,
    "co2_moy": 4081.6
  },
  "rural_isole__intermediaires": {
    "n": 7,
    "km_jour_moy": 50.7,
    "km_jour_med": 36.0,
    "km_jour_p75": 65.3,
    "km_jour_p90": 100.0,
    "nb_dep_moy": 3.0,
    "nb_dep_med": 2.0,
    "autonomie_moy": 22.9,
    "autonomie_med": 15.0,
    "autonomie_p75": 33.5,
    "autonomie_p90": 50.0,
    "co2_moy": 7586.1
  },
  "rural_isole__ouvriers": {
    "n": 14,
    "km_jour_moy": 37.7,
    "km_jour_med": 24.0,
    "km_jour_p75": 46.0,
    "km_jour_p90": 96.0,
    "nb_dep_moy": 4.1,
    "nb_dep_med": 4.0,
    "autonomie_moy": 10.3,
    "autonomie_med": 9.5,
    "autonomie_p75": 14.5,
    "autonomie_p90": 18.5,
    "co2_moy": 4949.9
  },
  "rural_isole__retraites": {
    "n": 98,
    "km_jour_moy": 31.3,
    "km_jour_med": 23.4,
    "km_jour_p75": 45.0,
    "km_jour_p90": 60.3,
    "nb_dep_moy": 3.2,
    "nb_dep_med": 2.0,
    "autonomie_moy": 14.8,
    "autonomie_med": 10.0,
    "autonomie_p75": 20.0,
    "autonomie_p90": 27.2,
    "co2_moy": 3538.8
  },
  "urbain_dense__artisans": {
    "n": 38,
    "km_jour_moy": 34.0,
    "km_jour_med": 15.8,
    "km_jour_p75": 34.5,
    "km_jour_p90": 83.8,
    "nb_dep_moy": 3.9,
    "nb_dep_med": 3.0,
    "autonomie_moy": 16.2,
    "autonomie_med": 7.6,
    "autonomie_p75": 15.7,
    "autonomie_p90": 45.1,
    "co2_moy": 3484.4
  },
  "urbain_dense__autre": {
    "n": 107,
    "km_jour_moy": 11.0,
    "km_jour_med": 4.0,
    "km_jour_p75": 11.1,
    "km_jour_p90": 20.0,
    "nb_dep_moy": 3.3,
    "nb_dep_med": 3.0,
    "autonomie_moy": 5.2,
    "autonomie_med": 1.5,
    "autonomie_p75": 4.8,
    "autonomie_p90": 9.4,
    "co2_moy": 413.2
  },
  "urbain_dense__cadres": {
    "n": 70,
    "km_jour_moy": 53.2,
    "km_jour_med": 14.7,
    "km_jour_p75": 31.6,
    "km_jour_p90": 68.1,
    "nb_dep_moy": 3.8,
    "nb_dep_med": 4.0,
    "autonomie_moy": 37.6,
    "autonomie_med": 6.8,
    "autonomie_p75": 14.8,
    "autonomie_p90": 25.5,
    "co2_moy": 1428.0
  },
  "urbain_dense__employes": {
    "n": 326,
    "km_jour_moy": 28.9,
    "km_jour_med": 13.6,
    "km_jour_p75": 27.9,
    "km_jour_p90": 45.1,
    "nb_dep_moy": 3.4,
    "nb_dep_med": 3.0,
    "autonomie_moy": 17.2,
    "autonomie_med": 6.0,
    "autonomie_p75": 11.7,
    "autonomie_p90": 21.0,
    "co2_moy": 1634.8
  },
  "urbain_dense__inactifs": {
    "n": 332,
    "km_jour_moy": 20.3,
    "km_jour_med": 8.0,
    "km_jour_p75": 18.0,
    "km_jour_p90": 35.7,
    "nb_dep_moy": 3.3,
    "nb_dep_med": 2.0,
    "autonomie_moy": 12.1,
    "autonomie_med": 3.6,
    "autonomie_p75": 7.5,
    "autonomie_p90": 18.0,
    "co2_moy": 1027.0
  },
  "urbain_dense__intermediaires": {
    "n": 186,
    "km_jour_moy": 33.5,
    "km_jour_med": 17.8,
    "km_jour_p75": 40.0,
    "km_jour_p90": 85.0,
    "nb_dep_moy": 3.8,
    "nb_dep_med": 4.0,
    "autonomie_moy": 16.5,
    "autonomie_med": 8.0,
    "autonomie_p75": 15.0,
    "autonomie_p90": 50.0,
    "co2_moy": 3001.2
  },
  "urbain_dense__ouvriers": {
    "n": 206,
    "km_jour_moy": 24.9,
    "km_jour_med": 15.7,
    "km_jour_p75": 28.8,
    "km_jour_p90": 63.7,
    "nb_dep_moy": 3.4,
    "nb_dep_med": 3.0,
    "autonomie_moy": 11.6,
    "autonomie_med": 6.0,
    "autonomie_p75": 12.0,
    "autonomie_p90": 25.2,
    "co2_moy": 2125.1
  },
  "urbain_dense__retraites": {
    "n": 585,
    "km_jour_moy": 19.1,
    "km_jour_med": 7.1,
    "km_jour_p75": 16.0,
    "km_jour_p90": 36.0,
    "nb_dep_moy": 3.1,
    "nb_dep_med": 2.0,
    "autonomie_moy": 11.1,
    "autonomie_med": 3.0,
    "autonomie_p75": 7.0,
    "autonomie_p90": 16.0,
    "co2_moy": 1229.2
  },
  "__fallbacks__": {
    "rural": {
      "km_jour_moy": 40,
      "km_jour_med": 30,
      "km_jour_p75": 58,
      "km_jour_p90": 90,
      "nb_dep_moy": 3.6,
      "nb_dep_med": 4,
      "autonomie_moy": 22,
      "autonomie_med": 18,
      "autonomie_p75": 30,
      "autonomie_p90": 52,
      "co2_moy": 4500,
      "n": 0
    },
    "periurbain": {
      "km_jour_moy": 32,
      "km_jour_med": 18,
      "km_jour_p75": 42,
      "km_jour_p90": 80,
      "nb_dep_moy": 3.5,
      "nb_dep_med": 3,
      "autonomie_moy": 17,
      "autonomie_med": 12,
      "autonomie_p75": 26,
      "autonomie_p90": 45,
      "co2_moy": 3000,
      "n": 0
    },
    "rural_isole": {
      "km_jour_moy": 38,
      "km_jour_med": 24,
      "km_jour_p75": 55,
      "km_jour_p90": 85,
      "nb_dep_moy": 3.4,
      "nb_dep_med": 3,
      "autonomie_moy": 18,
      "autonomie_med": 14,
      "autonomie_p75": 28,
      "autonomie_p90": 50,
      "co2_moy": 4200,
      "n": 0
    }
  }
}

FALLBACKS = EMP_REF.pop('__fallbacks__', {})

def get_emp_segment(territoire, csp):
    key = f'{territoire}__{csp}'
    if key in EMP_REF and EMP_REF[key]['n'] >= 5:
        return EMP_REF[key]
    # Fallback : moyenne du territoire
    terr_segments = {k:v for k,v in EMP_REF.items() if k.startswith(f'{territoire}__')}
    if terr_segments:
        # Moyenne pondérée par n
        total_n = sum(v['n'] for v in terr_segments.values() if v['n'] > 0)
        if total_n == 0: return FALLBACKS.get(territoire, FALLBACKS['rural'])
        result = {}
        num_keys = ['km_jour_moy','km_jour_med','km_jour_p75','km_jour_p90',
                    'nb_dep_moy','nb_dep_med','autonomie_moy','autonomie_med',
                    'autonomie_p75','autonomie_p90','co2_moy']
        for k in num_keys:
            result[k] = sum(v[k]*v['n'] for v in terr_segments.values() if v['n']>0) / total_n
        result['n'] = total_n
        return result
    return FALLBACKS.get(territoire, FALLBACKS['rural'])

# Test
seg = get_emp_segment('rural', 'ouvriers')
print(f'rural__ouvriers (n={seg["n"]}) :')
print(f'  km/jour médian : {seg["km_jour_med"]} km')
print(f'  Autonomie P75  : {seg["autonomie_p75"]} km  ← besoin batterie 75e centile')
print(f'  CO₂ actuel/jour : {seg["co2_moy"]:.0f} gCO₂  (si remplacé par véhicule élec → 0)')

In [ ]:
# @title 🗺️ 4. Mapping département → territoire + CSP Nemotron → CSP EMP
DEPT_ILE       = {'971','972','973','974','976','2A','2B'}
DEPT_MONTAGNE  = {'04','05','07','09','15','19','23','38','42','43','48','63','65','66','73','74'}
DEPT_URBAIN    = {'06','13','31','33','34','44','57','59','67','69','75','76','92','93','94'}
# Élargi pour capturer plus de territoires périurbains (grandes couronnes)
DEPT_PERIURBAIN= {'01','02','03','08','10','14','16','17','18','19','21','22','24','25',
                  '27','28','29','32','35','36','37','38','39','41','45','46','47','49',
                  '51','52','53','54','55','56','57','58','60','61','62','63','64','65',
                  '67','68','70','71','72','77','78','79','80','81','82','84','85','86',
                  '87','88','89','90','91','93','94','95'}

def get_territoire(dep):
    d = str(dep or '').strip()
    if d in DEPT_ILE:         return 'ile'
    if d in DEPT_MONTAGNE:    return 'rural_isole'
    if d in DEPT_URBAIN:      return 'urbain_dense'
    if d in DEPT_PERIURBAIN:  return 'periurbain'
    return 'rural'

LOW_INC_KW  = ['ouvrier','ouvrière','employé','employée','caissier','caissière','vendeur',
               'vendeuse','manutentionnaire','agent','aide','opérateur','technicien',
               'technicienne','sans emploi','chômage','rsa','retraité','retraitée',
               'apprenti','alternant','intérimaire','artisan','agriculteur','aide-soignant',
               'agricultrice','éleveur','pêcheur','livreur','chauffeur']
HIGH_INC_KW = ['directeur','directrice','ingénieur chef','médecin','avocat','cadre supérieur',
               'chirurgien','pdg','président','associé','gérant','notaire','pharmacien']

def is_low_income(occ):
    ol = (occ or '').lower()
    if any(k in ol for k in HIGH_INC_KW): return False
    if any(k in ol for k in LOW_INC_KW):  return True
    return True  # Par défaut inclure les indéterminés dans la cible

OCC_TO_CSP = [
    (['agriculteur','agricultrice','éleveur','pêcheur','vigneron','maraîcher'],   'agriculteurs'),
    (['artisan','boulanger','boucher','coiffeur','plombier','électricien',
      'maçon','menuisier','carreleur','peintre bâtiment','garagiste'],             'artisans'),
    (['ouvrier','ouvrière','manutentionnaire','opérateur','agent de production',
      'cariste','mécanicien usine','soudeur','chaudronnier'],                      'ouvriers'),
    (['employé','employée','caissier','caissière','vendeur','vendeuse',
      'secrétaire','agent administratif','aide-soignant','infirmier auxiliaire',
      'livreur','chauffeur VTC','magasinier'],                                     'employes'),
    (['technicien','technicienne','infirmier','infirmière','instituteur',
      'enseignant','comptable','assistant','animateur'],                           'intermediaires'),
    (['retraité','retraitée'],                                                     'retraites'),
    (['sans emploi','chômage','rsa','allocataire','intérimaire',
      'apprenti','alternant','stagiaire','étudiant'],                              'inactifs'),
]

def occ_to_csp(occ):
    ol = (occ or '').lower()
    for keywords, csp in OCC_TO_CSP:
        if any(k in ol for k in keywords): return csp
    return 'employes'  # fallback

# Tests
for test in ['Ouvrière sur chaîne de production','Livreur indépendant','Retraité fonctionnaire',
             'Technicienne de maintenance','Artisan boulanger']:
    print(f'{test:40} → {occ_to_csp(test)}')

In [ ]:
# @title 🎲 5. Fonctions d'attribution des stats mobilité avec bruit réaliste
import numpy as np

def assign_mobility(territoire, csp, age, household, rng):
    """
    Attribue des stats de mobilité individuelles à un persona
    en tirant dans la distribution EMP du segment correspondant.
    Bruit log-normal calibré sur la dispersion observée dans EMP.
    """
    seg = get_emp_segment(territoire, csp)

    # Bruit log-normal (sigma calibré sur rapport p75/médiane EMP)
    def lognorm_sample(median, p75, rng):
        if median <= 0 or p75 <= 0: return max(median, 1)
        # Estimer sigma à partir du rapport p75/médiane
        ratio = max(p75 / max(median, 0.1), 1.01)
        sigma = np.log(ratio) / 0.6745  # 0.6745 = Phi^-1(0.75)
        mu    = np.log(max(median, 0.1))
        return round(float(np.exp(rng.normal(mu, sigma))), 1)

    km_jour      = lognorm_sample(seg['km_jour_med'],   seg['km_jour_p75'],   rng)
    autonomie_km = lognorm_sample(seg['autonomie_med'], seg['autonomie_p75'],  rng)
    nb_dep       = max(1, int(round(rng.normal(seg['nb_dep_moy'], 0.8))))

    # Modulations démographiques (cohérentes EMP)
    try: age_v = int(age)
    except: age_v = 40
    if age_v < 25:           km_jour *= 0.75; autonomie_km *= 0.80
    elif age_v > 65:         km_jour *= 0.65; autonomie_km *= 0.65; nb_dep = max(1,nb_dep-1)
    elif 35 <= age_v <= 55:  km_jour *= 1.10; autonomie_km *= 1.05

    # Famille avec enfants → +15% km (accompagnements scolaires)
    hh = (household or '').lower()
    if any(k in hh for k in ['enfant','monoparent','famille']):
        km_jour     *= 1.15
        autonomie_km = max(autonomie_km, 8)  # minimum 8km si enfants

    # CO₂ économisé si véhicule électrique remplace la voiture
    # (proportionnel au km/jour, ratio EMP)
    co2_ratio  = seg['co2_moy'] / max(seg['km_jour_moy'], 1)
    co2_actuel = round(km_jour * co2_ratio)

    # Vitesse minimale requise selon territoire (EMP + logique routière)
    vitesse_min = {'urbain_dense':25,'periurbain':45,'rural':90,
                   'rural_isole':90,'ile':45}.get(territoire, 45)

    # Besoin autonomie = max(trajet_max × 1.15 de marge, km_jour × 0.7)
    # Le facteur 1.15 = marge de sécurité batterie
    besoin_autonomie = round(max(autonomie_km * 1.15, km_jour * 0.5))

    # Places nécessaires
    if any(k in hh for k in ['4 personnes','trois enfants','nombreuse']): places = 4
    elif any(k in hh for k in ['enfant','monoparent','famille']):          places = 3
    elif any(k in hh for k in ['couple','concubin','marié']):              places = 2
    else:                                                                    places = 1

    return {
        'emp_segment':      f'{territoire}__{csp}',
        'km_jour':          round(min(km_jour, 300), 1),
        'nb_deplacements':  min(nb_dep, 10),
        'dist_max_trajet':  round(min(autonomie_km, 200), 1),
        'besoin_autonomie_km': min(besoin_autonomie, 200),
        'vitesse_min_kmh':   vitesse_min,
        'places_besoin':     places,
        'co2_actuel_g_jour': co2_actuel,
        'co2_economie_g_jour': co2_actuel,  # économie si tout-électrique
    }

# Test
rng = np.random.default_rng(42)
for test in [('rural','ouvriers',43,'famille avec enfants'),
             ('periurbain','employes',27,'personne seule'),
             ('rural_isole','retraites',70,'couple sans enfant')]:
    r = assign_mobility(*test, rng)
    print(f'{test[0]:15} {test[1]:15} {test[2]}a → '
          f'{r["km_jour"]} km/j · {r["nb_deplacements"]} dép · '
          f'autonomie besoin {r["besoin_autonomie_km"]} km · '
          f'vitesse min {r["vitesse_min_kmh"]} km/h · '
          f'{r["places_besoin"]} place(s)')

In [ ]:
# @title 📥 6. Streaming Nemotron + enrichissement mobilité (200 000 personas)
from datasets import load_dataset
from tqdm.auto import tqdm
import pandas as pd, numpy as np

rng = np.random.default_rng(SEED)
dataset = load_dataset('nvidia/Nemotron-Personas-France', split='train', streaming=True)

# Compteurs de diagnostic
terr_counts = {}  # pour vérifier la répartition territoire avant filtrage
li_count = 0  # low-income count

results = []
scanned = 0
LOW_INC_TERR = {'periurbain','rural','rural_isole'}

for row in tqdm(dataset, desc='Scan Nemotron', total=SCAN_MAX):
    if scanned >= SCAN_MAX or len(results) >= N_TARGET:
        break
    scanned += 1

    occ       = row.get('occupation','') or ''
    dep       = str(row.get('departement','') or '')
    territoire = get_territoire(dep)

    if not is_low_income(occ):        continue
    if territoire not in LOW_INC_TERR: continue

    try: age = int(row.get('age', 40))
    except: age = 40
    if not (16 <= age <= 80): continue

    household = row.get('household_type','') or ''
    csp       = occ_to_csp(occ)
    mob       = assign_mobility(territoire, csp, age, household, rng)

    # Budget d'achat estimé
    csp_budget = {
        'ouvriers':8000,'employes':8500,'artisans':11000,'intermediaires':10000,
        'retraites':9000,'inactifs':4000,'agriculteurs':9500,'autre':8000
    }
    budget = csp_budget.get(csp, 8000)
    if 40 <= age <= 55: budget = int(budget * 1.12)
    elif age < 25:      budget = int(budget * 0.70)
    elif age > 65:      budget = int(budget * 0.85)

    results.append({
        # Identité
        'uuid':         row.get('uuid',''),
        'prenom':       str(row.get('persona',''))[:50],
        'age':          age,
        'sex':          row.get('sex',''),
        'commune':      row.get('commune',''),
        'departement':  dep,
        'territoire':   territoire,
        'occupation':   occ[:80],
        'csp':          csp,
        'household_type': household[:60],
        'education_level': row.get('education_level',''),
        # Profil enrichi
        'sport':        str(row.get('sports_persona',''))[:150],
        'travel':       str(row.get('travel_persona',''))[:150],
        'skills':       str(row.get('skills_and_expertise',''))[:150],
        'hobbies':      str(row.get('hobbies_and_interests',''))[:150],
        # Budget
        'budget_achat_eur': budget,
        # Mobilité (EMP 2019 calibré)
        **mob,
    })

print(f'\n✅ {len(results):,} personas collectés / {scanned:,} scannés')
df = pd.DataFrame(results)
print(f'\nRépartition territoire :')
print(df['territoire'].value_counts())
print(f'\nRépartition CSP :')
print(df['csp'].value_counts())
print(f'\nStats mobilité (médiane) :')
for col in ['km_jour','nb_deplacements','dist_max_trajet','besoin_autonomie_km','vitesse_min_kmh','places_besoin']:
    print(f'  {col}: {df[col].median()}')

In [ ]:
# @title 📊 7. Validation des distributions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distributions mobilité — 200k personas low-income rural/périurbain', fontsize=14)

plots = [
    ('km_jour',            'km/jour', 'blue',   [0,150]),
    ('dist_max_trajet',    'km trajet max', 'orange', [0,100]),
    ('besoin_autonomie_km','km autonomie requis', 'green',  [0,150]),
    ('nb_deplacements',    'nb déplacements/j', 'purple', [0,10]),
    ('budget_achat_eur',   'budget achat (€)', 'red',    [0,20000]),
    ('co2_actuel_g_jour',  'gCO₂/jour (actuel)', 'gray',   [0,15000]),
]

for ax, (col, label, color, xlim) in zip(axes.flat, plots):
    data = df[col].clip(*xlim)
    ax.hist(data, bins=40, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5,
               label=f'médiane={data.median():.0f}')
    ax.set_title(label); ax.set_xlabel(label); ax.legend(fontsize=9)
    ax.set_xlim(*xlim)

plt.tight_layout()
plt.savefig('distributions_mobilite_200k.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé')

In [ ]:
# @title 💾 8. Export CSV + téléchargement
from google.colab import files
import os

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
size_mb = os.path.getsize(OUTPUT_FILE) / 1_048_576

print(f'✅ Export : {OUTPUT_FILE}')
print(f'   {len(df):,} personas · {size_mb:.1f} Mo · {len(df.columns)} colonnes')
print(f'\nColonnes exportées :')
for i, col in enumerate(df.columns): print(f'  {i+1:2}. {col}')

print(f'\nStats synthétiques clés :')
print(f'  km/jour médian        : {df["km_jour"].median():.0f} km')
print(f'  Autonomie P50 requis  : {df["besoin_autonomie_km"].median():.0f} km')
print(f'  Autonomie P75 requis  : {df["besoin_autonomie_km"].quantile(.75):.0f} km')
print(f'  Autonomie P90 requis  : {df["besoin_autonomie_km"].quantile(.90):.0f} km')
print(f'  % besoin vitesse ≥90  : {(df["vitesse_min_kmh"]>=90).mean()*100:.0f}%')
print(f'  Budget médian         : {df["budget_achat_eur"].median():,.0f} €')
print(f'  CO₂ économisable/j    : {df["co2_economie_g_jour"].mean():.0f} gCO₂/pers (si 100% élec)')

files.download(OUTPUT_FILE)
print('📥 Téléchargement lancé !')

## ✅ Résultat

Le CSV `personas_mobilite_200k.csv` contient **200 000 personas** avec :

| Colonne | Source | Description |
|---|---|---|
| `km_jour` | EMP 2019 calibré | km parcourus par jour (log-normal) |
| `nb_deplacements` | EMP 2019 calibré | Nombre de déplacements par jour |
| `dist_max_trajet` | EMP 2019 calibré | Distance du trajet le plus long |
| `besoin_autonomie_km` | Calculé | Autonomie minimale requise (P75 + marge 15 %) |
| `vitesse_min_kmh` | Logique routière | 25/45/90 km/h selon territoire |
| `places_besoin` | Nemotron household | 1 à 4 places selon composition foyer |
| `budget_achat_eur` | Estimé par CSP+âge | Capacité d'achat véhicule |
| `co2_actuel_g_jour` | EMP 2019 ratio | CO₂ actuel si voiture thermique |
| `co2_economie_g_jour` | Identique | CO₂ économisé si 100 % électrique |

**À faire ensuite :** Uploader ce CSV dans le notebook `colab_reverse_6M.ipynb` ou directement dans l'app Streamlit `app_optimiseur.py` pour recalibrer les distributions.

## 🔁 Étape 9 — Distributions pour l'onglet Optimiseur des aides

Cette étape calcule les **distributions cumulatives** sur les vrais personas Nemotron,
pour remplacer les estimations EMP 2019 / CGDD dans l'onglet 1 de l'app.

**Profils assouplis** : les critères de matching ont été simplifiés pour éviter d'avoir
des profils vides (notamment les profils périurbains qui étaient trop restrictifs).

**Gestion automatique dans l'app** : si un profil a moins de 5% de couverture dans
les distributions JSON, l'app conserve automatiquement les valeurs EMP 2019 pour ce profil
et affiche un avertissement. Vous n'avez rien à faire de particulier.

| Profil | Critère simplifié | Note |
|---|---|---|
| `periurbain_cycliste` | Périurbain · solo · besoin ≤ 25 km | VAE cargo |
| `periurbain_solo` | Périurbain · solo · besoin > 25 km | L6e |
| `periurbain_famille` | Périurbain · ≥ 2 places | L6e |
| `rural_navetteur` | Rural · ≤ 2 places · < 60 km | L7e |
| `rural_famille` | Rural · ≥ 3 places | L7e |
| `rural_longue_dist` | Rural · ≥ 60 km · ≤ 2 places | L8e |

In [ ]:
# @title 📊 9a. Vérification du DataFrame avant calcul
import pandas as pd

# Vérifier les colonnes nécessaires
required = ['territoire','csp','besoin_autonomie_km','budget_achat_eur','places_besoin']
missing = [c for c in required if c not in df.columns]
if missing:
    print(f'❌ Colonnes manquantes : {missing}')
    print(f'   Colonnes disponibles : {list(df.columns)}')
else:
    print(f'✅ Toutes les colonnes nécessaires sont présentes')
    print(f'   {len(df):,} personas · territoires : {df["territoire"].value_counts().to_dict()}')
    print()
    print('Distribution besoin_autonomie_km (global) :')
    for p in [25,50,75,90]:
        print(f'  P{p} : {df["besoin_autonomie_km"].quantile(p/100):.0f} km')
    print()
    print('Distribution budget_achat_eur (global) :')
    for p in [25,50,75,90]:
        print(f'  P{p} : {df["budget_achat_eur"].quantile(p/100):,.0f} €')

In [ ]:
# @title 📊 9b. Définition des 6 profils archétypaux
# Les profils sont définis par territoire + critères simples sur les besoins
# IMPORTANT : ne pas filtrer sur vitesse_min_kmh car elle est déterminée par le territoire
# (tous les périurbains ont vitesse_min=45, tous les ruraux ont vitesse_min=90)

LOW_INC_TERR = ['periurbain', 'rural', 'rural_isole']

def profil_periurbain_cycliste(row):
    """Périurbain solo avec très courts trajets — candidat naturel au VAE cargo."""
    return (row['territoire'] == 'periurbain'
            and row['places_besoin'] == 1
            and row['besoin_autonomie_km'] <= 25)  # Trajets courts ≤ 25 km

def profil_periurbain_solo(row):
    """Périurbain solo avec trajets moyens."""
    return (row['territoire'] == 'periurbain'
            and row['places_besoin'] == 1
            and row['besoin_autonomie_km'] > 25)

def profil_periurbain_famille(row):
    """Périurbain avec enfants ou plusieurs adultes."""
    return (row['territoire'] == 'periurbain'
            and row['places_besoin'] >= 2)

def profil_rural_navetteur(row):
    """Rural ou isolé, petit foyer, autonomie modérée."""
    return (row['territoire'] in ['rural','rural_isole']
            and row['places_besoin'] <= 2
            and row['besoin_autonomie_km'] < 60)

def profil_rural_famille(row):
    """Rural ou isolé avec plusieurs places."""
    return (row['territoire'] in ['rural','rural_isole']
            and row['places_besoin'] >= 3)

def profil_rural_longue_dist(row):
    """Rural ou isolé avec longs trajets quotidiens (> 60 km aller-retour)."""
    return (row['territoire'] in ['rural','rural_isole']
            and row['besoin_autonomie_km'] >= 60
            and row['places_besoin'] <= 2)

PROFILE_MAP = {
    'periurbain_cycliste': profil_periurbain_cycliste,
    'periurbain_solo':     profil_periurbain_solo,
    'periurbain_famille':  profil_periurbain_famille,
    'rural_navetteur':     profil_rural_navetteur,
    'rural_famille':       profil_rural_famille,
    'rural_longue_dist':   profil_rural_longue_dist,
}

# Seuils pour les distributions
AUTO_KEYS   = [10, 20, 30, 40, 50, 60, 80, 100, 120, 150]
BUDGET_KEYS = [2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000,
               10000, 11000, 12000, 13000, 14000, 15000, 18000, 22000]

# Assigner chaque persona à son profil (ordre = priorité de matching)
def assign_profile(row):
    for pid, fn in PROFILE_MAP.items():
        if fn(row): return pid
    return 'hors_cible'

print('Attribution des 6 profils...')
df['profil_optimiseur'] = df.apply(assign_profile, axis=1)
counts = df['profil_optimiseur'].value_counts()
print(counts)
n_total = len(df[df['profil_optimiseur'] != 'hors_cible'])
pct = (df['profil_optimiseur'] != 'hors_cible').mean() * 100
print(f'\n{pct:.1f}% des personas appartiennent à un profil ({n_total:,} / {len(df):,})')

# Avertissement si profil periurbain trop petit
for pid in ['periurbain_cycliste', 'periurbain_solo', 'periurbain_famille']:
    n = counts.get(pid, 0)
    if n < 100:
        print(f"⚠️  {pid} : seulement {n} personas — les valeurs EMP 2019 seront conservées dans l'app")
    else:
        print(f"✅  {pid} : {n} personas — distribution calculable")


In [ ]:
# @title 📊 9c. Calcul des distributions cumulatives réelles (6 profils)
# Le profil 'periurbain_cycliste' sera inclus dans le JSON si présent dans les personas.
# Si sa taille est < 50, ses valeurs restent proches des hardcoded dans app.py.

import numpy as np

distributions = {
    'CUMUL_AUTO':      {},
    'AFFORD_PROFILES': {},
    'profil_sizes':    {},
    'profil_pct':      {},
}

n_total_cible = len(df[df['profil_optimiseur'] != 'hors_cible'])

print('='*65)
for pid in PROFILE_MAP.keys():
    sub = df[df['profil_optimiseur'] == pid]
    n = len(sub)
    pct = n / max(n_total_cible, 1)
    distributions['profil_sizes'][pid] = n
    distributions['profil_pct'][pid]   = round(pct, 4)

    # P(besoin_autonomie <= X km) — croissant avec X
    cumul_auto = {}
    for km in AUTO_KEYS:
        cumul_auto[km] = round(float((sub['besoin_autonomie_km'] <= km).mean()), 3)
    distributions['CUMUL_AUTO'][pid] = cumul_auto

    # P(budget >= prix) — décroissant avec le prix
    afford = {}
    for prix in BUDGET_KEYS:
        afford[prix] = round(float((sub['budget_achat_eur'] >= prix).mean()), 3)
    afford[99999] = 0.0
    distributions['AFFORD_PROFILES'][pid] = afford

    print(f'\n{pid} (n={n:,} · {pct*100:.1f}% de la cible) :')
    print(f'  Autonomie   P50={sub["besoin_autonomie_km"].median():.0f} '
          f'P75={sub["besoin_autonomie_km"].quantile(.75):.0f} '
          f'P90={sub["besoin_autonomie_km"].quantile(.90):.0f} km')
    print(f'  Budget      P50={sub["budget_achat_eur"].median():,.0f} '
          f'P75={sub["budget_achat_eur"].quantile(.75):,.0f} €')
    print(f'  P(auto≤30km)={cumul_auto.get(30,0)*100:.0f}%  '
          f'P(auto≤60km)={cumul_auto.get(60,0)*100:.0f}%  '
          f'P(budget≥8000€)={afford.get(8000,0)*100:.0f}%')

print('='*65)
print(f'\nTotal cible Nemotron réel : {n_total_cible:,} personas')

In [ ]:
# @title 💾 9d. Export distributions_nemotron.json → à charger dans Streamlit
import json
from google.colab import files

# Ajouter les métadonnées
distributions['n_cible_nemotron'] = n_total_cible
distributions['n_dataset_total']  = len(df)
distributions['source'] = (
    'nvidia/Nemotron-Personas-France CC-BY-4.0 × EMP 2019 SDES/INSEE'
)
distributions['description'] = (
    'Distributions cumulatives de besoin_autonomie et budget par profil archétypal. '
    'Calculées sur les personas Nemotron enrichis des données de mobilité EMP 2019. '
    'À charger dans l onglet Optimiseur de l app Streamlit via le bouton sidebar.'
)

output_file = 'distributions_nemotron.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(distributions, f, indent=2, ensure_ascii=False)

import os
size_kb = os.path.getsize(output_file) // 1024
print(f'✅ Fichier exporté : {output_file} ({size_kb} Ko)')
print(f'   Profils : {list(distributions["CUMUL_AUTO"].keys())} ({len(distributions["CUMUL_AUTO"])} total)')
# Avertissement si profils périurbains vides (résultat habituel avec filtre Q1+Q2)
n_periurbain = sum(distributions['profil_sizes'].get(p, 0)
                   for p in ['periurbain_cycliste','periurbain_solo','periurbain_famille'])
if n_periurbain == 0:
    print('\n⚠️  IMPORTANT : 0 personas périurbains trouvés dans cet échantillon.')
    print('   Raison probable : le filtre low-income Q1+Q2 sur Nemotron capture surtout des ruraux.')
    print('   → Ce n\'est PAS un bug. L\'app utilisera les valeurs EMP 2019 pour les profils périurbains.')
    print('   → Les résultats ruraux (navetteur, famille, longue_dist) sont valides et seront utilisés.')
    print('   Pour obtenir des périurbains : augmenter SCAN_MAX ou modifier is_low_income() dans cellule 5.')

if 'periurbain_cycliste' in distributions['CUMUL_AUTO']:
    n_cyc = distributions['profil_sizes'].get('periurbain_cycliste', 0)
    print(f'   ✅ Profil cycliste VAE inclus ({n_cyc:,} personas)')
else:
    print('   ⚠️ Profil cycliste absent — app.py utilisera les valeurs hardcoded')
print(f'   Population Nemotron réelle : {n_total_cible:,} personas')
print()
print('📥 Lancement du téléchargement...')
files.download(output_file)
print()
print('ÉTAPE SUIVANTE :')
print('1. Ouvrir l app Streamlit → onglet 🎯 Optimiseur véhicule')
print('2. Dans la sidebar gauche → "📥 Distributions Nemotron"')
print('3. Cliquer Browse files → sélectionner distributions_nemotron.json')
print('4. Le modèle se recalibre immédiatement sur les vrais personas Nemotron')

## ✅ Récapitulatif du flux complet

```
HuggingFace (6M personas Nemotron)
        ↓  Cellules 1-8 : stream + filtre + enrichissement EMP 2019
personas_mobilite_200k.csv  (200k lignes · mobilité réelle par persona)
        ↓  Cellules 9a-9d : calcul des 6 distributions cumulatives
distributions_nemotron.json  (CUMUL_AUTO + AFFORD_PROFILES · 6 profils)
        ↓  Upload dans sidebar onglet 3 Streamlit
Optimiseur recalibré — 5 véhicules optimisés sur les vrais personas Nemotron
```

### Ce que le JSON change dans l'optimiseur

| Donnée | Sans JSON (défaut) | Avec JSON Nemotron |
|---|---|---|
| Distributions autonomie | EMP 2019, 50-900 individus/profil | Milliers de personas Nemotron |
| Courbes budget | Calibration CGDD/INSEE estimée | Budget calculé par CSP × âge × décile INSEE |
| Population cible | 1,95M (estimation) | Décompte réel des personas filtrés |
| Parts des 6 profils | Estimées 8/12/25/28/20/7 % | Calculées sur la vraie distribution Nemotron |
| Profil cycliste VAE | Hardcodé dans app.py | Remplacé par données Nemotron si présent dans JSON |

### ⚠️ Si le profil cycliste est absent du JSON

Si le dataset Nemotron ne contient pas assez de personas répondant aux critères
du profil cycliste (solo + périurbain + besoin ≤ 20 km), le profil sera absent du JSON.
L'app utilisera alors les valeurs hardcodées dans `app.py` — ce qui est parfaitement acceptable
car ce profil est le plus incertain (représente une adoption future, pas un usage actuel mesuré).